### Day 06 - Sklearn & Regressions 
Setup & California Housing dataset

The California Housing dataset contains:

- **Features**: median income, house age, average rooms, average bedrooms, population, latitude, longitude, etc.
- **Target**: `MedHouseVal` — median house value in blocks of $100,000.

This is a regression problem: we predict a continuous numeric value.


### Regression vs Classification

**Classification** (Days 4–5):
- Target: discrete class labels (e.g. survived = 0 or 1).
- Metrics: accuracy, precision, recall, F1, ROC AUC.
- Models: Logistic Regression, RandomForestClassifier.

**Regression** (Day 6):
- Target: continuous numeric value (e.g. house price).
- Metrics: MAE, MSE, RMSE, R².
- Models: Linear Regression, Ridge, RandomForestRegressor.

Key difference:
- Classification predicts **which category**.
- Regression predicts **how much** or **how many**.


In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split

# Paths
notebook_path = Path.cwd()
repo_root = notebook_path
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

data_dir = repo_root / "data"
BostonHousing_path = data_dir / "BostonHousing.csv"

df = pd.read_csv(BostonHousing_path)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(3))
print("\nTarget (MEDV) statistics:")

print(df["medv"].describe())

# Define features and target
# Target: MEDV (median home value in $1000s)
# Features: all other columns
feature_cols = [col for col in df.columns if col != "medv"]
X = df[feature_cols]
y = df["medv"]

print("\nFeature columns:", feature_cols)
print("\nX shape:", X.shape)
print("y shape:", y.shape)

# Check for missing values
print("\nMissing values per column:")
print(df.isna().sum())

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("\nTrain shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

Shape: (506, 14)
Columns: ['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'b', 'lstat', 'medv']


,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7



Target (MEDV) statistics:
count    506.000000
mean      22.532806
std        9.197104
min        5.000000
25%       17.025000
50%       21.200000
75%       25.000000
max       50.000000
Name: medv, dtype: float64

Feature columns: ['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'b', 'lstat']

X shape: (506, 13)
y shape: (506,)

Missing values per column:
crim       0
zn         0
indus      0
chas       0
nox        0
rm         0
age        0
dis        0
rad        0
tax        0
ptratio    0
b          0
lstat      0
medv       0
dtype: int64

Train shape: (404, 13) (404,)
Test shape: (102, 13) (102,)


### Basic exploration and train/test split

We will:
- Inspect feature distributions.
- Check for missing values.
- Split into train (80%) and test (20%).


### Regression metrics

We evaluate regression models using:

1. **MAE (Mean Absolute Error)**
   - Average absolute difference between predicted and actual values.
   - Formula:
     MAE = (1/n) × sum of |actual - predicted| for all samples
   Example:  
Actual prices: 20, 25, 30 (in thousands)  
Predicted prices: 22, 23, 32
Errors: 2, -2, 2  
Absolute errors: 2, 2, 2
MAE = (2 + 2 + 2) / 3 = 2

2. **MSE (Mean Squared Error)**
   - Average squared difference.
   - Penalizes large errors more heavily.
   - Formula:
     MSE = (1/n) × sum of (actual - predicted)² for all samples
Example:  
Actual prices: 20, 25, 30  
Predicted prices: 22, 23, 32
Errors: 2, -2, 2  
Squared errors: 4, 4, 4
MSE = (4 + 4 + 4) / 3 = 4

3. **RMSE (Root Mean Squared Error)**
   - Square root of MSE.
   - Same units as the target.
   - Formula:
     \MSE = square root of MSE
Example:  
If MSE = 4
RMSE = √4 = 2

4. **R² (R-squared)**
   - Proportion of variance in the target explained by the model.
   - Range: typically 0 to 1.
   - R² = 1 means perfect predictions.
   - Formula:
   R² = 1 - (sum of squared errors from model / sum of squared errors from predicting the average)

   Example:  
If your model’s errors sum to 20 (squared)  
And predicting the average would give errors summing to 100 (squared)
R² = 1 - (20/100) = 1 - 0.2 = 0.8
Your model explains 80% of the variation in prices.




In [17]:
"""
Baseline: Linear Regression on Boston Housing.
"""

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Train a simple linear regression model
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# Predictions
y_pred_train = lin_reg.predict(X_train)
y_pred_test = lin_reg.predict(X_test)

# Evaluate
def evaluate_regression(y_true, y_pred, dataset_name):
    """Calculate and print regression metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    print(f"\n{dataset_name} metrics:")
    print(f"  MAE:  {mae:.4f}")
    print(f"  MSE:  {mse:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R²:   {r2:.4f}")

evaluate_regression(y_train, y_pred_train, "Train")
evaluate_regression(y_test, y_pred_test, "Test")

# Inspect coefficients
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": lin_reg.coef_
}).sort_values("coefficient", key=abs, ascending=False)

print("\nLinear Regression coefficients:")
display(coef_df)



Train metrics:
  MAE:  3.3148
  MSE:  21.6414
  RMSE: 4.6520
  R²:   0.7509

Test metrics:
  MAE:  3.1891
  MSE:  24.2911
  RMSE: 4.9286
  R²:   0.6688

Linear Regression coefficients:


,feature,coefficient
4,nox,-17.202633
5,rm,4.438835
3,chas,2.784438
7,dis,-1.447865
10,ptratio,-0.915456
12,lstat,-0.508571
8,rad,0.262430
0,crim,-0.113056
2,indus,0.040381
1,zn,0.030110


### Linear Regression - 
Definition:  
Finds the best straight-line relationship between features and target.
Formula:  
predicted = w₀ + w₁×feature₁ + w₂×feature₂ + … + w₁₃×feature₁₃
Example for Boston Housing:  
Let’s say the model learns:
•	w₀ (intercept) = 10
•	w_rm (coefficient for rooms) = 5
•	w_lstat (coefficient for lower status) = -0.5
For a house with 6 rooms and 10% lower status:
predicted_price = 10 + 5×6 + (-0.5)×10 = 10 + 30 - 5 = 35 thousand dollars

### Ridge Regression -
Definition:  
Linear Regression with a penalty for large coefficients to prevent overfitting.
Formula:  
Minimize: sum of squared errors + alpha × sum of squared coefficients
Example:  
Two models with same training error:
Model A (no regularization):  
Coefficients: 10, 5, -8, 3, … (some very large)
Model B (Ridge with alpha=10):  
Coefficients: 2, 1, -1, 0.5, … (all smaller)
Ridge prefers Model B because large coefficients are penalized. This usually generalizes better to new data.

### Ridge Regression (L2 Regularization)

**What it is:**

Ridge Regression is a variant of Linear Regression that adds **L2 regularization** to prevent overfitting.

**How it differs from Linear Regression:**

- Linear Regression minimizes:
  \\[\\sum\_{i=1}^{n} (y\_i - \\hat{y}\_i)^2\\]

- Ridge Regression minimizes:
  \\[\\sum\_{i=1}^{n} (y\_i - \\hat{y}\_i)^2 + \\alpha \\sum\_{j=1}^{p} w\_j^2\\]

  The second term (\\(\\alpha \\sum w\_j^2\\)) is the **regularization penalty**.

**The `alpha` parameter:**

- Controls the strength of regularization.
- **Large `alpha`** → strong regularization → coefficients are pushed closer to zero → simpler model.
- **Small `alpha`** → weak regularization → model can fit training data more closely.
- `alpha = 0` is equivalent to ordinary Linear Regression.

**Why use Ridge?**

- Prevents overfitting, especially when:
  - You have many features.
  - Features are highly correlated (multicollinearity).
- Shrinks coefficients but doesn't eliminate them (unlike Lasso).
- Often improves generalization compared to ordinary Linear Regression.

**When to use:**

- When Linear Regression overfits.
- When you have correlated features.
- When you want a simple model but better generalization.


In [18]:
"""
Ridge Regression with different alpha values.
"""

from sklearn.linear_model import Ridge

# Test different alpha values
alphas = [0.1, 1, 10, 100]

print("Ridge Regression - Test set performance:")
for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train, y_train)
    y_pred = ridge.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"  alpha={alpha:4.1f} → MAE: {mae:.4f}, R²: {r2:.4f}")


Ridge Regression - Test set performance:
  alpha= 0.1 → MAE: 3.1789, R²: 0.6686
  alpha= 1.0 → MAE: 3.1329, R²: 0.6662
  alpha=10.0 → MAE: 3.1251, R²: 0.6639
  alpha=100.0 → MAE: 3.1739, R²: 0.6800


### Random Forest Regressor - Non Linear Model

Builds many decision trees (like 100 trees). Each tree:
•	Sees a random subset of the training data
•	Considers a random subset of features at each split
Final prediction = average of all tree predictions.
Pros:
•	Handles non-linear relationships well
•	Doesn’t need feature scaling
•	Robust to outliers
Cons:
•	Less interpretable than linear models
•	Slower

### Random Forest Regressor (Ensemble Method)

**What it is:**

Random Forest is an **ensemble learning** method that combines many decision trees to make predictions.

**How it works:**

1. **Bootstrap sampling**: Creates multiple random subsets of the training data (with replacement).
2. **Train many trees**: Each tree is trained on a different subset.
3. **Random feature selection**: At each split in a tree, only a random subset of features is considered.
4. **Average predictions**: For regression, the final prediction is the average of all tree predictions.

**Why "Random" Forest?**

- Randomness comes from:
  - Different training data for each tree (bootstrap samples).
  - Different features considered at each split.
- This randomness makes trees diverse and reduces overfitting.

**Pros:**

- Can capture non-linear relationships and interactions between features.
- Generally more accurate than linear models on complex datasets.
- Robust to outliers and doesn't require feature scaling.
- Provides feature importance scores.

**Cons:**

- Less interpretable than linear models (can't easily explain individual coefficients).
- Slower to train and predict than linear models.
- Can still overfit if trees are too deep or too many.

**Key hyperparameters:**

- `n_estimators`: Number of trees in the forest (more trees → more stable, but slower).
- `max_depth`: Maximum depth of each tree (deeper trees → more complex, risk of overfitting).
- `min_samples_leaf`: Minimum samples required in a leaf node (higher values → simpler trees).

**When to use:**

- When linear models underperform.
- When you suspect non-linear relationships.
- When you want a strong baseline without extensive tuning.


This Averages predictions from many decision trees, each trained on random subsets of data and features.
Formula:  
Final prediction = average of all tree predictions
Example:  
5 trees predict: 30, 32, 28, 35, 31 (thousand dollars)
Final prediction = (30 + 32 + 28 + 35 + 31) / 5 = 156 / 5 = 31.2 thousand dollars
Each tree might overfit differently, but averaging reduces overall error.

In [19]:
"""
Random Forest Regressor on Boston Housing.
"""

from sklearn.ensemble import RandomForestRegressor

# Train a Random Forest
rf_reg = RandomForestRegressor(
    n_estimators=100,       # Number of trees
    max_depth=None,         # No limit on tree depth
    random_state=42,
    n_jobs=-1               # Use all CPU cores
)

rf_reg.fit(X_train, y_train)

# Evaluate
y_pred_rf = rf_reg.predict(X_test)
evaluate_regression(y_test, y_pred_rf, "Random Forest - Test")

# Feature importances
importances_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_reg.feature_importances_
}).sort_values("importance", ascending=False)

print("\nRandom Forest feature importances:")
display(importances_df.head(10))



Random Forest - Test metrics:
  MAE:  2.0489
  MSE:  8.1157
  RMSE: 2.8488
  R²:   0.8893

Random Forest feature importances:


,feature,importance
5,rm,0.503800
12,lstat,0.309730
7,dis,0.059623
0,crim,0.036567
10,ptratio,0.018203
9,tax,0.016607
4,nox,0.014911
6,age,0.013442
11,b,0.012701
2,indus,0.007365


### Cross-Validation for Regression

**What is cross-validation?**

Cross-validation (CV) is a technique to evaluate model performance more reliably than a single train/test split.

**How k-fold CV works:**

1. Split the data into k equal parts (folds).
2. For each fold:
   - Train the model on k-1 folds.
   - Evaluate on the held-out fold.
3. Average the performance across all k folds.

**Why use cross-validation?**

- A single train/test split can be lucky or unlucky.
- CV gives a more stable estimate of how the model will perform on unseen data.
- Helps detect overfitting.

**For regression:**

- We use `KFold` or `ShuffleSplit` (not stratified, since the target is continuous).
- Common metrics: MAE, MSE, R².
- scikit-learn returns negative MAE/MSE in CV (because it maximizes scores), so we convert them back to positive.

**Typical choices:**

- k = 5 or k = 10 folds.
- Shuffle the data before splitting to avoid order bias.

Definition:  
Splits data into k parts, trains on k-1 parts, tests on 1 part. Repeats k times.
Example with 5-fold CV:  
Data: 1000 houses
Fold 1: Train on 800, test on 200 → R² = 0.75  
Fold 2: Train on 800, test on 200 → R² = 0.78  
Fold 3: Train on 800, test on 200 → R² = 0.72  
Fold 4: Train on 800, test on 200 → R² = 0.76  
Fold 5: Train on 800, test on 200 → R² = 0.74
Average R² = (0.75 + 0.78 + 0.72 + 0.76 + 0.74) / 5 = 0.75
More reliable than a single train/test split.

In [20]:
"""
5-fold cross-validation for regression models.
"""

from sklearn.model_selection import cross_validate, KFold

# Define 5-fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Metrics to evaluate
scoring = ["neg_mean_absolute_error", "neg_mean_squared_error", "r2"]

# Cross-validate Linear Regression
cv_results_lin = cross_validate(
    LinearRegression(),
    X, y,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

# Cross-validate Random Forest
cv_results_rf = cross_validate(
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    X, y,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

# Summarize results
def summarize_cv_results(results, model_name):
    """Print mean and std for each CV metric."""
    print(f"\n{model_name} - 5-fold CV results:")
    for metric in scoring:
        key = f"test_{metric}"
        mean_val = results[key].mean()
        std_val = results[key].std()
        # Convert negative MAE/MSE back to positive for readability
        if metric.startswith("neg_"):
            mean_val = -mean_val
        print(f"  {metric}: {mean_val:.4f} ± {std_val:.4f}")

summarize_cv_results(cv_results_lin, "Linear Regression")
summarize_cv_results(cv_results_rf, "Random Forest")



Linear Regression - 5-fold CV results:
  neg_mean_absolute_error: 3.3885 ± 0.1873
  neg_mean_squared_error: 23.4886 ± 1.8426
  r2: 0.7152 ± 0.0375

Random Forest - 5-fold CV results:
  neg_mean_absolute_error: 2.1887 ± 0.2035
  neg_mean_squared_error: 10.6824 ± 3.2165
  r2: 0.8721 ± 0.0329


### Hyperparameter Tuning with GridSearchCV

**What are hyperparameters?**

Hyperparameters are settings you choose before training, not learned from data.

Examples for Random Forest:
- `n_estimators`: How many trees to build.
- `max_depth`: How deep each tree can grow.
- `min_samples_leaf`: Minimum samples in a leaf node.

**What is GridSearchCV?**

GridSearchCV systematically tries all combinations of hyperparameters you specify:

1. Defines a "grid" of parameter values to test.
2. For each combination:
   - Uses cross-validation to evaluate performance.
3. Selects the combination with the best average CV score.
4. Refits the model with the best parameters on the full training data.

**Why use GridSearchCV?**

- Avoids manual trial-and-error.
- Ensures you're testing hyperparameters fairly using CV.
- Helps find a good balance between underfitting and overfitting.

**Important:**

- Always tune on training data only.
- Keep the test set completely separate for final evaluation.
- Grid search can be slow with many parameters (consider `RandomizedSearchCV` for large grids).

Definition:  
Tries all combinations of specified hyperparameters using cross-validation, picks the best.
Example:  
Testing Random Forest with:
•	n_estimators: 50, 100
•	max_depth: None, 5
Combinations to test:
1.	50 trees, no depth limit → CV R² = 0.72
2.	50 trees, max_depth=5 → CV R² = 0.75
3.	100 trees, no depth limit → CV R² = 0.74
4.	100 trees, max_depth=5 → CV R² = 0.78
Best: 100 trees, max_depth=5 (R² = 0.78)
GridSearchCV picks this combination and retrains on all training data.

In [21]:
"""
GridSearchCV for Random Forest hyperparameter tuning.
"""

from sklearn.model_selection import GridSearchCV

# Parameter grid
param_grid = {
    "n_estimators": [50, 100, 200],      # Number of trees
    "max_depth": [None, 5, 10],          # Maximum tree depth
    "min_samples_leaf": [1, 5, 10]       # Minimum samples per leaf
}

# Grid search
grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid,
    scoring="neg_mean_absolute_error",   # Minimize MAE
    cv=cv,                                # Use 5-fold CV from Section 5
    n_jobs=-1,                            # Parallelize across CPU cores
    return_train_score=False
)

grid_search.fit(X_train, y_train)

print("Best CV MAE:", round(-grid_search.best_score_, 4))
print("Best params:", grid_search.best_params_)

# Evaluate best model on test set
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
evaluate_regression(y_test, y_pred_best, "Best Random Forest - Test")


Best CV MAE: 2.3957
Best params: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}

Best Random Forest - Test metrics:
  MAE:  2.0597
  MSE:  8.7787
  RMSE: 2.9629
  R²:   0.8803


### Pipeline with Preprocessing

**What is a pipeline?**

A pipeline chains preprocessing steps and a model into a single object.

**Why use pipelines?**

- Ensures preprocessing is applied consistently to train and test data.
- Prevents data leakage (preprocessing is fit only on training data).
- Makes code cleaner and easier to maintain.
- Works seamlessly with `GridSearchCV` and `cross_validate`.

**In this example:**

- We add `StandardScaler` to scale features to mean 0, std 1.
- Random Forest doesn't require scaling, but this demonstrates the pattern.
- For datasets with missing values or categorical features, pipelines become essential.


In [22]:
"""
Pipeline with StandardScaler + Random Forest.
"""

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Pipeline with scaling + Random Forest
pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

pipeline.fit(X_train, y_train)
y_pred_pipe = pipeline.predict(X_test)

evaluate_regression(y_test, y_pred_pipe, "Pipeline (Scaled + RF) - Test")



Pipeline (Scaled + RF) - Test metrics:
  MAE:  2.0503
  MSE:  8.1253
  RMSE: 2.8505
  R²:   0.8892
